# bigcompute.science MCP Explorer

**22 tools for computational mathematics research.** Query experiments, cross-reference findings against arXiv/zbMATH/OEIS, search Lean 4/Mathlib, and explore open problems — all from this notebook.

No API key needed. No setup. Just run the cells.

- Website: [bigcompute.science](https://bigcompute.science)
- Audit Ledger: [bigcompute.science/verification](https://bigcompute.science/verification/)
- Source: [github.com/cahlen/idontknow](https://github.com/cahlen/idontknow)
- License: CC BY 4.0

In [ ]:
# === Setup: MCP Client (run this first) ===
import requests, json

MCP = "https://mcp.bigcompute.science/mcp"

# Initialize connection
requests.post(MCP, json={
    "jsonrpc": "2.0", "id": 0,
    "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "colab-explorer", "version": "1.0"}}
})

def mcp(tool, **kwargs):
    """Call any MCP tool. Returns parsed JSON."""
    r = requests.post(MCP, json={
        "jsonrpc": "2.0", "id": 1,
        "method": "tools/call",
        "params": {"name": tool, "arguments": kwargs}
    })
    text = r.json().get("result", {}).get("content", [{}])[0].get("text", "{}")
    try: return json.loads(text)
    except: return text

def pp(data):
    """Pretty print JSON."""
    print(json.dumps(data, indent=2) if isinstance(data, (dict, list)) else data)

print("Connected to bigcompute.science MCP (22 tools)")
print("Usage: mcp('tool_name', param1=value1, param2=value2)")

## Explore the Data

Start by listing what's available:

In [ ]:
# List all experiments
experiments = mcp('list_experiments')
for e in experiments:
    print(f"[{e['status']:12s}] {e['title'][:70]}")

In [ ]:
# List all findings with audit levels
findings = mcp('list_findings')
for f in findings:
    cert = f.get('certification', {})
    level = cert.get('level', '?').upper()
    print(f"[{level:6s}] {f['title'][:60]}")

## Cross-Reference Against Literature

Pick any finding and verify it against arXiv, zbMATH, and OEIS:

In [ ]:
# Verify the Zaremba density finding against live academic databases
result = mcp('verify_finding', finding='zaremba-density')

print(f"Finding: {result['finding']['title']}")
print(f"Claim: {result['finding']['claim'][:150]}...")
print()
for src in ['arxiv', 'zbmath', 'oeis']:
    items = result['literature'].get(src, [])
    if items:
        print(f"{src.upper()} ({len(items)} results):")
        for p in items[:3]:
            name = p.get('title', p.get('name', p.get('id', '?')))
            print(f"  - {name[:70]}")
        print()

## Search Academic Databases

Search arXiv, zbMATH, OEIS, LMFDB, and Lean 4/Mathlib directly:

In [ ]:
# Search arXiv for papers on Zaremba's conjecture
papers = mcp('search_arxiv', query='Zaremba conjecture continued fractions', max_results=5)
for p in papers:
    print(f"[{p['published']}] {p['title']}")
    print(f"  {p['url']}")
    print()

In [ ]:
# Look up a sequence in OEIS
seqs = mcp('lookup_oeis', query='1,1,2,3,5,8,13')
for s in seqs[:3]:
    print(f"{s['id']}: {s['name']}")
    print(f"  {s.get('data', '')[:60]}")
    print()

In [ ]:
# Search zbMATH (peer-reviewed mathematics)
papers = mcp('search_zbmath', query='Hausdorff dimension continued fraction Cantor set', max_results=3)
for p in papers.get('papers', []):
    print(f"[{p.get('year', '?')}] {p['title']} by {p.get('authors', '?')}")
    print()

In [ ]:
# Search Lean 4 / Mathlib theorems
results = mcp('search_mathlib', query='Nat.Prime')
if isinstance(results, dict) and 'results' in results:
    print(f"Found {results.get('count', '?')} results:")
    for r in results['results'][:5]:
        print(f"  {r['name']} : {r.get('type', '?')[:50]}")
else:
    print(results)  # May return error if Loogle blocks Cloudflare

## Get Details on Any Experiment

Get CUDA kernel paths, compile commands, and reproduction instructions:

In [ ]:
# Get full details on the Zaremba verification experiment
exp = mcp('get_experiment', slug='zaremba-conjecture-verification')
pp(exp)

In [ ]:
# Get CUDA kernel compile + run commands
kernel = mcp('get_cuda_kernel', experiment='zaremba-density')
pp(kernel)

## What Should I Compute Next?

Get experiment recommendations based on available GPU time:

In [ ]:
# What experiments are most valuable right now?
suggestions = mcp('suggest_experiment', gpu_hours=10, interest='any')
for s in suggestions.get('recommended', []):
    print(f"#{s['priority']} {s['experiment']}")
    print(f"   {s['impact'][:80]}")
    print(f"   Command: {s.get('command', 'N/A')}")
    print()

## The Zaremba Exceptions

The 27 integers that have no coprime fraction with all CF partial quotients ≤ 3:

In [ ]:
# Get the famous 27 exceptions
exc = mcp('get_zaremba_exceptions')
print(f"Digit set: {exc['digit_set']}")
print(f"Verified to: {exc['verified_to']}")
print(f"Total exceptions: {exc['total_exceptions']}")
print(f"All below: {exc['all_below']}")
print(f"\nExceptions: {exc['exceptions']}")
print(f"\nHierarchy:")
for k, v in exc['hierarchy'].items():
    print(f"  {k}: {v}")

## All Available Tools

Here's every tool you can call with `mcp('tool_name', ...)`:

In [ ]:
# List all 22 tools
r = requests.post(MCP, json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}})
tools = r.json()['result']['tools']
print(f"{len(tools)} tools available:\n")
for t in tools:
    print(f"  mcp('{t['name']}', ...)")
    print(f"    {t['description'][:70]}")
    print()

## How the Audit System Works

In [ ]:
# Learn about the AI audit process
process = mcp('get_certification_process')
if isinstance(process, dict):
    print(process.get('process', '')[:500])
    print("\nCurrent certifications:")
    for c in process.get('current', []):
        print(f"  [{c['level'].upper():6s}] {c['slug']}")
else:
    print(process)

---

**bigcompute.science** — Guerrilla Mathematics™

AI-audited, not peer-reviewed. All code and data open. CC BY 4.0.

If you find something interesting, [submit a review](https://github.com/cahlen/idontknow/blob/main/docs/verifications/SCHEMA.md) or [contribute an experiment](https://github.com/cahlen/idontknow/blob/main/AGENTS.md).